# Triumph speedometer — first training experiment

**500 grayscale images: 400 training + 100 synthetic validation.** This notebook trains a pretrained MobileNetV3-Small with a speed output and two needle keypoint heatmaps. It compares nested training sets of **100, 250, and 400 images**, while keeping the same validation set.

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another available GPU).
2. Choose **Runtime → Run all**.
3. When the upload control appears, select **`speedometer_bw500.zip`**. The notebook already contains all training code.

**Output:** best checkpoints, speed/keypoint metrics, prediction tables, learning curves, worst-case overlays, and a results ZIP. The ZIP is offered for download after training. Keep it before disconnecting the runtime.

The proposed target is **98% of real readable frames within ±2 km/h of the verified dial reading**. This notebook reports the same tolerance on **synthetic development validation only**. Its 100 validation images represent 50 paired poses from the same reconstructed source; they do not establish live-camera accuracy. Real footage has not been supplied.

Sources: [Colab runtime FAQ](https://research.google.com/colaboratory/faq.html), [MobileNetV3-Small pretrained weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v3_small.html).

## 1. Check the runtime

The default run requires a GPU. Colab's existing PyTorch installation is used without replacing its CUDA packages. A separate smoke mode exists for local software checks; smoke results are **not** model performance.

In [ ]:
import os, sys, json, time, hashlib, zipfile, csv, importlib.util
from pathlib import Path
import torch, torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as DisplayImage

IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
RUN_MODE = os.environ.get('SPEEDOMETER_RUN_MODE', 'pilot')
assert RUN_MODE in ('pilot', 'smoke')
WORK_DIR = Path(os.environ.get('SPEEDOMETER_WORK_DIR', '/content/speedometer_work' if IN_COLAB else str(Path.cwd() / 'speedometer_work')))
WORK_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path(os.environ.get('SPEEDOMETER_DATASET_ZIP', '/content/speedometer_bw500.zip' if IN_COLAB else str(Path.cwd() / 'speedometer_bw500.zip')))
EXPECTED_ZIP_SHA256 = '92e773ec1ccee15a455a78453e8e5cfe81d04e2d93a57ff83fcea807b6ab3e27'
EXPECTED_LABELS_SHA256 = 'e77ea6f7354597008bec89b9c98acf8eca03ca76310921c53b2c822f1d2ef195'

print('Mode:', RUN_MODE, '| Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not connected')
if not hasattr(torch.amp, 'GradScaler'):
    raise RuntimeError('This notebook needs PyTorch 2.3 or newer; reconnect to a current Colab runtime.')
if RUN_MODE == 'pilot' and not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, reconnect, then Run all. No CPU pilot was started.')
if RUN_MODE == 'smoke':
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    print('SMOKE CHECK ONLY: short random-weight run, not an accuracy experiment.')

## 2. Upload and verify the dataset

Choose the supplied **`speedometer_bw500.zip`**, not one of the earlier 20-image archives. All 500 model inputs and the original split labels will be checked. Captions and preview images are excluded from training.

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

if not ZIP_PATH.is_file():
    if not IN_COLAB:
        raise FileNotFoundError('Set SPEEDOMETER_DATASET_ZIP to the supplied dataset ZIP.')
    from google.colab import files
    print('Select speedometer_bw500.zip (approximately 101 MB).')
    uploaded = files.upload()
    candidates = [Path(name).resolve() for name in uploaded if name.lower().endswith('.zip')]
    if len(candidates) != 1:
        raise ValueError('Upload exactly one dataset ZIP, then rerun this cell.')
    ZIP_PATH = candidates[0]
    del uploaded

if file_sha256(ZIP_PATH) != EXPECTED_ZIP_SHA256:
    raise ValueError('This is not the verified 500-image archive. Select the supplied speedometer_bw500.zip.')

with zipfile.ZipFile(ZIP_PATH) as archive:
    if archive.testzip() is not None:
        raise ValueError('The dataset archive failed its integrity check.')
    for entry in archive.infolist():
        target = (WORK_DIR / entry.filename).resolve()
        if not target.is_relative_to(WORK_DIR.resolve()) or not entry.filename.startswith('bw_dataset_500/'):
            raise ValueError('Unexpected path in the dataset archive.')
    archive.extractall(WORK_DIR)

DATA_DIR = WORK_DIR / 'bw_dataset_500'
if file_sha256(DATA_DIR / 'labels.json') != EXPECTED_LABELS_SHA256:
    raise ValueError('Label provenance mismatch.')
print('Verified dataset archive:', ZIP_PATH.name)
display(DisplayImage(filename=str(DATA_DIR / 'overview.png'), width=700))

## 3. Load the training implementation

The complete source is embedded below and saved into the runtime. No extra code upload is needed. The model preserves the full 512 × 512 crop. Grayscale values are repeated into three channels to reuse pretrained image features; no color information is introduced.

Training uses speed regression plus supervised pivot/tip heatmaps. It does not convert the raw perspective-distorted needle angle into speed. Pretrained features are frozen initially, then the final blocks are fine-tuned. Batch-normalization statistics stay frozen for the small batches.

In [ ]:
#@title Load the embedded training module
TRAINING_CORE_SOURCE = '"""Reproducible first speedometer training pilot (PyTorch / Google Colab).\n\nRun from Python:\n    result = train_experiment(TrainConfig(data_dir="/content/bw_dataset_500",\n                                         output_dir="/content/run_400"))\nThis evaluates synthetic validation data, never a real-world test set. Input\nimages already contain camera and appearance augmentation. No additional image\ngeometry is changed here, so keypoint targets remain aligned with the pixels.\n"""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nimport math\nimport random\nimport time\nfrom collections import Counter, defaultdict\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small\n\n\n@dataclass\nclass TrainConfig:\n    data_dir: str\n    output_dir: str\n    input_size: int = 512\n    batch_size: int = 8\n    head_epochs: int = 5\n    finetune_epochs: int = 25\n    patience: int = 7\n    seed: int = 42\n    pretrained: bool = True\n    max_train_samples: int | None = None\n    max_val_samples: int | None = None\n    head_lr: float = 1e-3\n    finetune_lr: float = 1e-4\n    backbone_lr: float = 2e-5\n    weight_decay: float = 1e-4\n    num_workers: int = 0\n    amp: bool = True\n    device: str = "auto"\n    min_delta_kmh: float = 0.01\n    heatmap_weight: float = 0.2\n    coordinate_weight: float = 5.0\n    heatmap_sigma: float = 1.5\n\n\ndef _sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _json_write(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n\n\ndef _data_path(root, relative):\n    path = (Path(root) / relative).resolve()\n    if not path.is_relative_to(Path(root).resolve()):\n        raise ValueError(f"Dataset path is outside data_dir: {relative}")\n    return path\n\n\ndef verify_dataset(data_dir):\n    """Check every model input and label, plus pose-group split isolation."""\n    root = Path(data_dir)\n    label_path = root / "labels.json"\n    document = json.loads(label_path.read_text(encoding="utf-8"))\n    records = document["samples"]\n    if len(records) != document["count"]:\n        raise ValueError("labels.json sample count disagrees with metadata")\n    counts = Counter(row["dataset_split"] for row in records)\n    if dict(counts) != document["split_counts"]:\n        raise ValueError("Split counts disagree with labels.json metadata")\n    if set(counts) != {"train", "validation"}:\n        raise ValueError("Expected nonempty train and validation splits")\n    ids, paths, hashes, groups = set(), set(), set(), defaultdict(set)\n    for row in records:\n        sid, relative = row["sample_id"], row["image"]\n        if sid in ids or relative in paths:\n            raise ValueError(f"Duplicate sample ID or image path: {sid}")\n        ids.add(sid)\n        paths.add(relative)\n        image_path = _data_path(root, relative)\n        digest = _sha256(image_path)\n        if digest != row["image_sha256"]:\n            raise ValueError(f"Image checksum mismatch: {sid}")\n        if digest in hashes:\n            raise ValueError(f"Duplicate image content: {sid}")\n        hashes.add(digest)\n        with Image.open(image_path) as im:\n            if im.mode != "L" or im.size != (row["width"], row["height"]):\n                raise ValueError(f"Unexpected mode or dimensions: {sid}")\n        speed = float(row["speed_kmh"])\n        if not math.isfinite(speed) or not 0 <= speed <= 180:\n            raise ValueError(f"Invalid speed: {sid}")\n        for name in ("pivot", "tip"):\n            xy = np.asarray(row["needle_position_px"][name], dtype=float)\n            norm = np.asarray(row["needle_position_normalized"][name], dtype=float)\n            limits = np.array([row["width"] - 1, row["height"] - 1])\n            if xy.shape != (2,) or norm.shape != (2,):\n                raise ValueError(f"Invalid keypoint shape: {sid}/{name}")\n            if not np.isfinite(xy).all() or not np.all((xy >= 0) & (xy <= limits)):\n                raise ValueError(f"Out-of-frame keypoint: {sid}/{name}")\n            if not np.allclose(xy / limits, norm, atol=1e-6):\n                raise ValueError(f"Normalized keypoint mismatch: {sid}/{name}")\n        groups[row["group_id"]].add(row["dataset_split"])\n    leaking = [gid for gid, splits in groups.items() if len(splits) != 1]\n    if leaking:\n        raise ValueError(f"Pose groups cross splits: {leaking[:5]}")\n    source = document.get("source", {})\n    if source.get("file") and source.get("sha256"):\n        if _sha256(_data_path(root, source["file"])) != source["sha256"]:\n            raise ValueError("Approved source checksum mismatch")\n    report = {\n        "count": len(records), "split_counts": dict(counts),\n        "pose_groups": len(groups), "pose_group_leakage": False,\n        "all_image_hashes_verified": True, "labels_sha256": _sha256(label_path),\n        "source_sha256": source.get("sha256"),\n        "scope": "synthetic validation; no independent real test set",\n    }\n    return records, report\n\n\ndef select_group_subset(records, limit, seed):\n    """Deterministic nested subsets; never divide variants of a pose group.\n\n    Singleton approved examples precede shuffled paired groups. This makes the\n    100/250/400 learning-curve subsets nested and exact for this dataset.\n    A requested size that would divide a group is rounded down and reported.\n    """\n    groups = defaultdict(list)\n    for row in records:\n        groups[row["group_id"]].append(row)\n    ordered = sorted(groups)\n    rng = random.Random(seed)\n    rng.shuffle(ordered)\n    ordered.sort(key=lambda gid: len(groups[gid]))\n    limit = len(records) if limit is None else int(limit)\n    if limit <= 0:\n        raise ValueError("Subset size must be positive")\n    selected = []\n    for gid in ordered:\n        group = sorted(groups[gid], key=lambda row: row["sample_id"])\n        if len(selected) + len(group) > limit:\n            break\n        selected.extend(group)\n    if not selected:\n        raise ValueError("Subset limit is smaller than one complete pose group")\n    return selected\n\n\nclass DashboardDataset(Dataset):\n    def __init__(self, data_dir, records, input_size=512):\n        self.root, self.records, self.input_size = Path(data_dir), records, input_size\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, index):\n        row = self.records[index]\n        image = preprocess_image(self.root / row["image"], self.input_size)\n        points = torch.tensor([row["needle_position_normalized"][name]\n                               for name in ("pivot", "tip")], dtype=torch.float32)\n        return {"image": image, "speed": torch.tensor(float(row["speed_kmh"])),\n                "points": points, "index": index}\n\n\ndef preprocess_image(image, input_size=512):\n    """Accept a PIL image or image path; return normalized 3-channel CHW input.\n\n    Grayscale is repeated into RGB solely to reuse ImageNet-pretrained weights.\n    The caller must crop real camera footage to match the training dashboard.\n    """\n    if isinstance(image, (str, Path)):\n        with Image.open(image) as opened:\n            gray = opened.convert("L")\n    elif isinstance(image, Image.Image):\n        gray = image.convert("L")\n    else:\n        raise TypeError("image must be a PIL image or a file path")\n    gray = gray.resize((input_size, input_size), Image.Resampling.BILINEAR)\n    array = np.asarray(gray, dtype=np.float32).copy() / 255.0\n    tensor = torch.from_numpy(array).unsqueeze(0).repeat(3, 1, 1)\n    mean = tensor.new_tensor([0.485, 0.456, 0.406])[:, None, None]\n    std = tensor.new_tensor([0.229, 0.224, 0.225])[:, None, None]\n    return (tensor - mean) / std\n\n\ndef _conv_block(in_channels, out_channels):\n    return nn.Sequential(nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),\n                         nn.GroupNorm(8, out_channels), nn.SiLU(inplace=True))\n\n\ndef heatmap_coordinates(logits):\n    """Spatial soft-argmax, returning normalized (x,y) in pivot/tip order."""\n    batch, channels, height, width = logits.shape\n    probabilities = logits.float().flatten(2).softmax(-1).view(batch, channels, height, width)\n    xs = torch.linspace(0, 1, width, device=logits.device)\n    ys = torch.linspace(0, 1, height, device=logits.device)\n    x = (probabilities.sum(dim=2) * xs).sum(-1)\n    y = (probabilities.sum(dim=3) * ys).sum(-1)\n    return torch.stack((x, y), dim=-1)\n\n\nclass SpeedometerModel(nn.Module):\n    """MobileNetV3-Small encoder, spatial keypoint decoder, continuous speed head."""\n    def __init__(self, pretrained=True):\n        super().__init__()\n        try:\n            encoder = mobilenet_v3_small(\n                weights=MobileNet_V3_Small_Weights.DEFAULT if pretrained else None)\n        except Exception as exc:\n            if pretrained:\n                raise RuntimeError("Pretrained MobileNet weights could not be loaded. "\n                                   "Enable internet and retry; no random-weight fallback was used.") from exc\n            raise\n        self.features = encoder.features\n        # torchvision MobileNetV3-Small skip channels: 16@1/4, 24@1/8, 48@1/16.\n        self.lateral = _conv_block(576, 96)\n        self.decode16 = _conv_block(96 + 48, 64)\n        self.decode8 = _conv_block(64 + 24, 32)\n        self.decode4 = _conv_block(32 + 16, 16)\n        self.heatmap_head = nn.Conv2d(16, 2, 1)\n        self.speed_head = nn.Sequential(nn.Linear(576 + 6, 128), nn.SiLU(),\n                                        nn.Dropout(0.1), nn.Linear(128, 1))\n\n    def forward(self, image):\n        skips = {}\n        value = image\n        for index, layer in enumerate(self.features):\n            value = layer(value)\n            if index in (1, 3, 8):\n                skips[index] = value\n        deep = value\n        value = self.lateral(deep)\n        for index, block in ((8, self.decode16), (3, self.decode8), (1, self.decode4)):\n            value = F.interpolate(value, size=skips[index].shape[-2:],\n                                  mode="bilinear", align_corners=True)\n            value = block(torch.cat((value, skips[index]), dim=1))\n        logits = self.heatmap_head(value)\n        points = heatmap_coordinates(logits)\n        relative = points[:, 1] - points[:, 0]\n        pooled = F.adaptive_avg_pool2d(deep, 1).flatten(1)\n        speed_input = torch.cat((pooled, points.flatten(1), relative), dim=1)\n        speed = self.speed_head(speed_input).squeeze(1).sigmoid() * 180.0\n        return {"speed": speed, "points": points, "heatmap_logits": logits}\n\n    def configure_stage(self, stage):\n        if stage not in {"heads", "finetune"}:\n            raise ValueError(f"Unknown stage: {stage}")\n        for param in self.features.parameters():\n            param.requires_grad_(False)\n        if stage == "finetune":\n            for layer in list(self.features.children())[-4:]:\n                for param in layer.parameters():\n                    param.requires_grad_(True)\n        self.train()\n\n    def train(self, mode=True):\n        super().train(mode)\n        # Small batches should not overwrite pretrained running BN statistics.\n        # Affine BN parameters may still train in the final unfrozen blocks.\n        for module in self.features.modules():\n            if isinstance(module, nn.modules.batchnorm._BatchNorm):\n                module.eval()\n        return self\n\n\ndef training_loss(output, batch, config):\n    logits = output["heatmap_logits"].float()\n    height, width = logits.shape[-2:]\n    points = batch["points"].float()\n    yy, xx = torch.meshgrid(torch.arange(height, device=logits.device),\n                            torch.arange(width, device=logits.device), indexing="ij")\n    centers = points * points.new_tensor([width - 1, height - 1])\n    squared = ((xx[None, None] - centers[:, :, 0, None, None]) ** 2\n               + (yy[None, None] - centers[:, :, 1, None, None]) ** 2)\n    target = torch.exp(-squared / (2 * config.heatmap_sigma ** 2)).flatten(2)\n    target = target / target.sum(-1, keepdim=True).clamp_min(1e-12)\n    heatmap = -(target * logits.flatten(2).log_softmax(-1)).sum(-1).mean()\n    coordinate = F.smooth_l1_loss(output["points"], points, beta=0.02)\n    # A unit is 10 km/h and the Huber transition is 2 km/h; gradients stay modest.\n    speed = F.smooth_l1_loss(output["speed"].float() / 10,\n                           batch["speed"].float() / 10, beta=0.2)\n    total = speed + config.heatmap_weight * heatmap + config.coordinate_weight * coordinate\n    return total, {"speed_loss": float(speed.detach()),\n                   "heatmap_loss": float(heatmap.detach()),\n                   "coordinate_loss": float(coordinate.detach())}\n\n\ndef _device(config):\n    return torch.device("cuda" if torch.cuda.is_available() else "cpu") if config.device == "auto" else torch.device(config.device)\n\n\ndef _seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef _loader(dataset, config, shuffle=False):\n    generator = torch.Generator().manual_seed(config.seed)\n    return DataLoader(dataset, batch_size=config.batch_size, shuffle=shuffle,\n                      num_workers=config.num_workers, pin_memory=_device(config).type == "cuda",\n                      generator=generator, drop_last=False)\n\n\ndef metrics_from_rows(rows):\n    if not rows:\n        raise ValueError("Cannot evaluate an empty dataset")\n    errors = np.array([row["error_kmh"] for row in rows], dtype=float)\n    absolute = np.abs(errors)\n    return {\n        "n": len(rows), "pose_groups": len({row["group_id"] for row in rows}),\n        "mae_kmh": float(absolute.mean()),\n        "rmse_kmh": float(np.sqrt(np.mean(errors ** 2))),\n        "p95_absolute_error_kmh": float(np.percentile(absolute, 95)),\n        "max_absolute_error_kmh": float(absolute.max()),\n        "bias_kmh": float(errors.mean()),\n        **{f"within_{bound}_kmh_percent": float((absolute <= bound).mean() * 100)\n           for bound in (1, 2, 5)},\n        "pivot_mae_px": float(np.mean([row["pivot_error_px"] for row in rows])),\n        "tip_mae_px": float(np.mean([row["tip_error_px"] for row in rows])),\n        "keypoint_error_definition": "Mean Euclidean distance in original dataset-image pixels",\n        "scope": "synthetic validation only; not real-world test accuracy",\n    }\n\n\n@torch.inference_mode()\ndef evaluate_model(model, loader, device):\n    model.eval()\n    rows = []\n    for batch in loader:\n        output = model(batch["image"].to(device, non_blocking=True))\n        speeds = output["speed"].float().cpu().numpy()\n        points = output["points"].float().cpu().numpy()\n        for local, dataset_index in enumerate(batch["index"].tolist()):\n            record = loader.dataset.records[dataset_index]\n            predicted_speed = float(speeds[local])\n            row = {"sample_id": record["sample_id"], "group_id": record["group_id"],\n                   "image": record["image"], "dataset_split": record["dataset_split"],\n                   "profile": record.get("augmentation", {}).get("photometric", {}).get("profile", "unknown"),\n                   "inferred_60_influence": bool(record.get("inferred_60_influence", False)),\n                   "target_speed_kmh": float(record["speed_kmh"]),\n                   "predicted_speed_kmh": predicted_speed,\n                   "error_kmh": predicted_speed - float(record["speed_kmh"])}\n            row["absolute_error_kmh"] = abs(row["error_kmh"])\n            for index, name in enumerate(("pivot", "tip")):\n                predicted = points[local, index] * np.array([record["width"] - 1, record["height"] - 1])\n                target = np.array(record["needle_position_px"][name])\n                row[f"{name}_error_px"] = float(np.linalg.norm(predicted - target))\n                for dimension, axis in enumerate(("x", "y")):\n                    row[f"predicted_{name}_{axis}"] = float(predicted[dimension])\n                    row[f"target_{name}_{axis}"] = float(target[dimension])\n            rows.append(row)\n    return metrics_from_rows(rows), rows\n\n\ndef _save_csv(path, rows):\n    if not rows:\n        return\n    with Path(path).open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef save_evaluation_artifacts(rows, history, data_dir, output_dir):\n    """Save inspectable predictions, hardest examples and training curves."""\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    out = Path(output_dir)\n    _save_csv(out / "validation_predictions.csv", rows)\n    _save_csv(out / "history.csv", history)\n    hardest = sorted(rows, key=lambda row: row["absolute_error_kmh"], reverse=True)[:12]\n    plot_rows = max(1, (len(hardest) + 3) // 4)\n    figure, axes = plt.subplots(plot_rows, 4, figsize=(16, 4.2 * plot_rows + 0.6), squeeze=False)\n    for axis in axes.flat:\n        axis.axis("off")\n    for axis, row in zip(axes.flat, hardest):\n        with Image.open(Path(data_dir) / row["image"]) as im:\n            axis.imshow(im, cmap="gray", vmin=0, vmax=255)\n        for prefix, color, style in (("target", "lime", "-"), ("predicted", "magenta", "--")):\n            axis.plot([row[f"{prefix}_pivot_x"], row[f"{prefix}_tip_x"]],\n                      [row[f"{prefix}_pivot_y"], row[f"{prefix}_tip_y"]],\n                      style, color=color, linewidth=1.5, marker="o", markersize=3)\n        axis.set_title(f"{row[\'sample_id\']} | target {row[\'target_speed_kmh\']:.1f}\\n"\n                       f"pred {row[\'predicted_speed_kmh\']:.1f} | error {row[\'error_kmh\']:+.1f} km/h", fontsize=10)\n    figure.suptitle("Hardest synthetic validation samples — green: label; magenta: prediction")\n    figure.tight_layout(rect=(0, 0, 1, 0.94))\n    figure.savefig(out / "worst_predictions.png", dpi=130)\n    plt.close(figure)\n    figure, axes = plt.subplots(1, 2, figsize=(11, 4))\n    axes[0].plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label="Training loss")\n    axes[0].set(xlabel="Epoch", ylabel="Combined training loss")\n    axes[1].plot([h["epoch"] for h in history], [h["validation_mae_kmh"] for h in history])\n    axes[1].set(xlabel="Epoch", ylabel="Synthetic validation MAE (km/h)")\n    axes[1].ticklabel_format(axis="y", style="plain", useOffset=False)\n    for axis in axes:\n        axis.grid(alpha=0.25)\n    figure.tight_layout()\n    figure.savefig(out / "learning_curves.png", dpi=140)\n    plt.close(figure)\n\n\ndef _optimizer(model, config, stage):\n    heads = [p for name, p in model.named_parameters() if p.requires_grad and not name.startswith("features.")]\n    groups = [{"params": heads, "lr": config.head_lr if stage == "heads" else config.finetune_lr}]\n    backbone = [p for p in model.features.parameters() if p.requires_grad]\n    if backbone:\n        groups.append({"params": backbone, "lr": config.backbone_lr})\n    return torch.optim.AdamW(groups, weight_decay=config.weight_decay)\n\n\ndef train_experiment(config: TrainConfig):\n    """Train once, restore the best validation-MAE checkpoint and export reports.\n\n    max_*_samples and pretrained=False support CPU smoke tests; their metrics\n    must not be presented as trained model quality. Full runs use all 400/100.\n    """\n    if config.input_size < 64 or config.input_size % 32:\n        raise ValueError("input_size must be >= 64 and divisible by 32 (512 for the pilot)")\n    if min(config.head_epochs, config.finetune_epochs) < 0 or config.head_epochs + config.finetune_epochs < 1:\n        raise ValueError("At least one training epoch is required")\n    if config.batch_size < 1 or config.patience < 1 or config.heatmap_sigma <= 0:\n        raise ValueError("batch_size, patience and heatmap_sigma must be positive")\n    out = Path(config.output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    if (out / "best_model.pt").exists():\n        raise FileExistsError("Use a new output_dir to preserve the previous experiment")\n    _seed_everything(config.seed)\n    records, integrity = verify_dataset(config.data_dir)\n    train_records = select_group_subset([r for r in records if r["dataset_split"] == "train"],\n                                         config.max_train_samples, config.seed)\n    val_records = select_group_subset([r for r in records if r["dataset_split"] == "validation"],\n                                       config.max_val_samples, config.seed)\n    integrity.update({"used_train_samples": len(train_records), "used_validation_samples": len(val_records),\n                      "used_train_ids": [r["sample_id"] for r in train_records],\n                      "used_validation_ids": [r["sample_id"] for r in val_records]})\n    _json_write(out / "data_integrity.json", integrity)\n    _json_write(out / "config.json", asdict(config))\n    train_loader = _loader(DashboardDataset(config.data_dir, train_records, config.input_size), config, True)\n    val_loader = _loader(DashboardDataset(config.data_dir, val_records, config.input_size), config)\n    device = _device(config)\n    model = SpeedometerModel(pretrained=config.pretrained).to(device)\n    amp_enabled = config.amp and device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)\n    best_mae, best_epoch, epoch, history = float("inf"), 0, 0, []\n    start = time.time()\n    best_path = out / "best_model.pt"\n    print(f"Device: {device}; train={len(train_records)}, synthetic validation={len(val_records)}", flush=True)\n    for stage, epochs in (("heads", config.head_epochs), ("finetune", config.finetune_epochs)):\n        if not epochs:\n            continue\n        # Start fine-tuning from the best head-training checkpoint.\n        if stage == "finetune" and best_path.exists():\n            model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True)["model_state_dict"])\n        model.configure_stage(stage)\n        optimizer = _optimizer(model, config, stage)\n        stale = 0\n        stage_best = float("inf")\n        for _ in range(epochs):\n            epoch += 1\n            model.train()\n            totals = Counter()\n            seen = 0\n            for batch in train_loader:\n                batch = {key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value\n                         for key, value in batch.items()}\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=device.type, enabled=amp_enabled):\n                    output = model(batch["image"])\n                    loss, components = training_loss(output, batch, config)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError("Nonfinite training loss; no result was silently accepted")\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 5.0)\n                scaler.step(optimizer)\n                scaler.update()\n                count = len(batch["speed"])\n                seen += count\n                totals["train_loss"] += float(loss.detach()) * count\n                for name, value in components.items():\n                    totals[name] += value * count\n            metrics, _ = evaluate_model(model, val_loader, device)\n            mae = metrics["mae_kmh"]\n            if not math.isfinite(mae):\n                raise FloatingPointError("Nonfinite validation MAE")\n            history.append({"epoch": epoch, "stage": stage,\n                            **{name: value / seen for name, value in totals.items()},\n                            "validation_mae_kmh": mae,\n                            "validation_within_2_kmh_percent": metrics["within_2_kmh_percent"],\n                            "validation_tip_mae_px": metrics["tip_mae_px"]})\n            print(f"Epoch {epoch:02d} {stage:8s} loss={totals[\'train_loss\']/seen:.4f} "\n                  f"val MAE={mae:.3f} km/h; within2={metrics[\'within_2_kmh_percent\']:.1f}%", flush=True)\n            # Save the actual minimum, separately from the patience improvement threshold.\n            if mae < best_mae:\n                best_mae, best_epoch = mae, epoch\n                torch.save({"format_version": 1, "model_state_dict": model.state_dict(),\n                            "config": asdict(config), "best_epoch": best_epoch,\n                            "validation_metrics": metrics, "labels_sha256": integrity["labels_sha256"],\n                            "keypoint_order": ["pivot", "tip"], "speed_range_kmh": [0, 180],\n                            "preprocessing": "PIL grayscale -> square bilinear resize -> repeated RGB -> ImageNet norm",\n                            "scope": "synthetic pilot; real camera validation pending"}, best_path)\n            if mae < stage_best - config.min_delta_kmh:\n                stage_best, stale = mae, 0\n            else:\n                stale += 1\n            _save_csv(out / "history.csv", history)\n            if stale >= config.patience:\n                print(f"Early stopping {stage}: no material MAE improvement for {stale} epochs", flush=True)\n                break\n    model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True)["model_state_dict"])\n    metrics, rows = evaluate_model(model, val_loader, device)\n    profiles = {profile: metrics_from_rows([row for row in rows if row["profile"] == profile])\n                for profile in sorted({row["profile"] for row in rows})}\n    speed_bins = {}\n    for lower in range(0, 180, 20):\n        members = [row for row in rows if lower <= row["target_speed_kmh"] < lower + 20\n                   or (lower == 160 and row["target_speed_kmh"] == 180)]\n        if members:\n            speed_bins[f"{lower}_to_{lower + 20}_kmh"] = metrics_from_rows(members)\n    calibration_groups = {}\n    for influenced in (False, True):\n        members = [row for row in rows if row["inferred_60_influence"] == influenced]\n        if members:\n            calibration_groups["inferred_60_region" if influenced else "other_regions"] = metrics_from_rows(members)\n    report = {"metrics": metrics, "metrics_by_profile": profiles,\n              "metrics_by_speed_bin": speed_bins, "metrics_by_calibration_region": calibration_groups,\n              "speed_bin_convention": "20-km/h bins are lower-inclusive, upper-exclusive; final bin includes 180",\n              "best_epoch": best_epoch,\n              "epochs_run": epoch, "elapsed_seconds": time.time() - start, "device": str(device),\n              "pretrained": config.pretrained, "training_samples": len(train_records),\n              "validation_samples": len(val_records), "labels_sha256": integrity["labels_sha256"],\n              "torch_version": str(torch.__version__), "config": asdict(config),\n              "limitation": "Shared synthetic master: these are development validation results, "\n                            "not evidence of 98% real-world accuracy. No independent real test was run."}\n    _json_write(out / "metrics.json", report)\n    save_evaluation_artifacts(rows, history, config.data_dir, out)\n    return {**report, "checkpoint_path": str(best_path), "metrics_path": str(out / "metrics.json"),\n            "predictions_path": str(out / "validation_predictions.csv"),\n            "worst_predictions_path": str(out / "worst_predictions.png"),\n            "learning_curves_path": str(out / "learning_curves.png")}\n\n\ndef load_model(checkpoint_path, device="auto"):\n    """Load this module\'s tensor-only checkpoint without downloading weights."""\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device == "auto" else torch.device(device)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    model = SpeedometerModel(pretrained=False)\n    model.load_state_dict(checkpoint["model_state_dict"])\n    model.to(device).eval()\n    return model, checkpoint\n\n\n@torch.inference_mode()\ndef predict_image(model, image, input_size=512):\n    """Predict a matched dashboard crop; coordinates refer to resized input.\n\n    No confidence score or unreadability detector is claimed by this model.\n    """\n    device = next(model.parameters()).device\n    model.eval()\n    tensor = preprocess_image(image, input_size).unsqueeze(0).to(device)\n    result = model(tensor)\n    points = result["points"][0].float().cpu().numpy()\n    return {"speed_kmh": float(result["speed"][0]),\n            "keypoints_normalized": {name: points[index].tolist() for index, name in enumerate(("pivot", "tip"))},\n            "keypoints_input_px": {name: (points[index] * (input_size - 1)).tolist()\n                                   for index, name in enumerate(("pivot", "tip"))},\n            "input_size": input_size}\n'
EXPECTED_CORE_SHA256 = 'da55e80192feef0298ec713d9fed59402a260af42042b3c5f9e001c73311a663'

assert hashlib.sha256(TRAINING_CORE_SOURCE.encode('utf-8')).hexdigest() == EXPECTED_CORE_SHA256
module_path = WORK_DIR / 'train_core.py'
module_path.write_text(TRAINING_CORE_SOURCE, encoding='utf-8')
spec = importlib.util.spec_from_file_location('speedometer_core', module_path)
training = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = training
spec.loader.exec_module(training)
print('Training module loaded; source SHA256:', EXPECTED_CORE_SHA256)

In [ ]:
records, integrity = training.verify_dataset(DATA_DIR)
assert integrity['split_counts'] == {'train': 400, 'validation': 100}
train_records = [r for r in records if r['dataset_split'] == 'train']
validation_records = [r for r in records if r['dataset_split'] == 'validation']
assert len({r['group_id'] for r in validation_records}) == 50
assert not ({r['speed_kmh'] for r in train_records} & {r['speed_kmh'] for r in validation_records})

SEED = 42
previous_ids = set()
subset_table = []
for size in (100, 250, 400):
    subset = training.select_group_subset(train_records, size, SEED)
    ids = {r['sample_id'] for r in subset}
    assert len(subset) == size and previous_ids.issubset(ids)
    previous_ids = ids
    subset_table.append({'training_images': size, 'training_poses': len({r['group_id'] for r in subset}),
                         'validation_images': 100, 'validation_poses': 50})
display(pd.DataFrame(subset_table))
print('All input hashes, keypoint coordinates, and split groups verified.')
print('Validation speed range:', min(r['speed_kmh'] for r in validation_records), 'to', max(r['speed_kmh'] for r in validation_records), 'km/h')
print('Exact 0 and 180 km/h examples are training-only; the validation set does not test those exact endpoints.')

## 4. Train the pilot

Default: **100, 250, and 400 training images**, 512-pixel input, batch size 8, up to 5 frozen-feature epochs followed by 25 fine-tuning epochs. Each experiment starts from the same pretrained weights and seed. The same 100 validation images are used throughout, and the best checkpoint is selected by validation mean absolute speed error.

To run just the full-data experiment, set `RUN_LEARNING_CURVE = False`. If the GPU reports an out-of-memory error, lower `BATCH_SIZE` to 4 and rerun this section. Each execution creates a new output directory; previous checkpoints are preserved. Availability and runtime depend on the assigned Colab GPU.

In [ ]:
RUN_LEARNING_CURVE = True
BATCH_SIZE = 8
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 25
PATIENCE = 7

from datetime import datetime, timezone
run_stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
SESSION_DIR = WORK_DIR / ('smoke_' if RUN_MODE == 'smoke' else 'pilot_') / run_stamp
SESSION_DIR.mkdir(parents=True, exist_ok=False)
training_sizes = [100, 250, 400] if RUN_LEARNING_CURVE else [400]
if RUN_MODE == 'smoke':
    training_sizes = [8]

results = []
for size in training_sizes:
    config = training.TrainConfig(
        data_dir=str(DATA_DIR), output_dir=str(SESSION_DIR / f'train_{size}'),
        input_size=128 if RUN_MODE == 'smoke' else 512,
        batch_size=2 if RUN_MODE == 'smoke' else BATCH_SIZE,
        head_epochs=1 if RUN_MODE == 'smoke' else HEAD_EPOCHS,
        finetune_epochs=1 if RUN_MODE == 'smoke' else FINETUNE_EPOCHS,
        patience=PATIENCE, seed=SEED,
        pretrained=(RUN_MODE != 'smoke'), max_train_samples=size,
        max_val_samples=4 if RUN_MODE == 'smoke' else None,
    )
    print()
    print(f'Starting {size}-image experiment')
    result = training.train_experiment(config)
    result['run_mode'] = RUN_MODE
    results.append(result)
    print('Saved:', result['checkpoint_path'])
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

(SESSION_DIR / 'experiment_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
print('All requested experiments finished:', SESSION_DIR)

## 5. Compare results and inspect mistakes

**Within ±2 km/h** is the proportion of validation predictions whose absolute speed error is at most 2 km/h. MAE is the average absolute speed error. Also inspect the 95th-percentile error and the needle tip error.

These are raw single-frame predictions, before temporal smoothing. No learned confidence or unreadable-frame detector is claimed. Paired images share pose groups, so the 100 validation frames are not 100 independent real-world trials. Reports also separate appearance profiles, speed bins, and the inferred 60-tick region.

In [ ]:
comparison_rows = []
for result in results:
    m = result['metrics']
    comparison_rows.append({
        'training_images': result['training_samples'], 'validation_images': result['validation_samples'],
        'validation_poses': m['pose_groups'], 'MAE_kmh': m['mae_kmh'],
        'RMSE_kmh': m['rmse_kmh'], 'within_2_kmh_percent': m['within_2_kmh_percent'],
        'P95_error_kmh': m['p95_absolute_error_kmh'], 'max_error_kmh': m['max_absolute_error_kmh'],
        'tip_error_px': m['tip_mae_px'], 'best_epoch': result['best_epoch'],
        'epochs_run': result['epochs_run'], 'run_mode': RUN_MODE,
    })
comparison = pd.DataFrame(comparison_rows).sort_values('training_images')
comparison.to_csv(SESSION_DIR / 'learning_curve_comparison.csv', index=False)
display(comparison)
if RUN_MODE == 'smoke':
    print('SMOKE RESULTS ONLY — verifies execution, not speed-reading accuracy.')
else:
    print('SYNTHETIC VALIDATION ONLY — real-camera accuracy remains unmeasured.')

selected_result = next((r for r in results if r['training_samples'] == 400), results[-1])
display(DisplayImage(filename=selected_result['learning_curves_path']))
display(DisplayImage(filename=selected_result['worst_predictions_path'], width=1100))
display(pd.DataFrame(selected_result['metrics_by_profile']).T)

## 6. Check a saved model and export the results

This cell reloads the selected checkpoint and predicts one validation image using the same preprocessing. Green shows the supplied needle label; magenta shows the model's prediction. Later, actual camera crops must match the training framing, and real speed labels are needed for evaluation.

In [ ]:
saved_model, checkpoint = training.load_model(selected_result['checkpoint_path'])
example = validation_records[0]
prediction = training.predict_image(saved_model, DATA_DIR / example['image'], input_size=checkpoint['config']['input_size'])
with Image.open(DATA_DIR / example['image']) as im:
    example_pixels = np.asarray(im)
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(example_pixels, cmap='gray', vmin=0, vmax=255)
for source, color, style in [(example['needle_position_normalized'], 'lime', '-'),
                             (prediction['keypoints_normalized'], 'magenta', '--')]:
    points = np.array([source['pivot'], source['tip']]) * 511
    ax.plot(points[:, 0], points[:, 1], style, color=color, marker='o', linewidth=1.5)
ax.set_title(f"Label {example['speed_kmh']:.1f} km/h | prediction {prediction['speed_kmh']:.1f} km/h")
ax.axis('off')
fig.tight_layout()
fig.savefig(SESSION_DIR / 'reloaded_model_example.png', dpi=130)
plt.close(fig)
display(DisplayImage(filename=str(SESSION_DIR / 'reloaded_model_example.png')))

In [ ]:
import shutil
shutil.copyfile(module_path, SESSION_DIR / 'train_core.py')
export_note = {
    'run_mode': RUN_MODE, 'dataset_labels_sha256': EXPECTED_LABELS_SHA256,
    'training_core_sha256': EXPECTED_CORE_SHA256,
    'scope': 'Synthetic development validation, not real-camera testing',
    'real_accuracy_measured': False,
    'next_decision': 'Inspect errors and learning curves; collect independently labeled real footage before deployment.',
}
(SESSION_DIR / 'run_scope.json').write_text(json.dumps(export_note, indent=2), encoding='utf-8')
RESULTS_ZIP = Path(shutil.make_archive(str(SESSION_DIR) + '_results', 'zip', root_dir=SESSION_DIR))
print('Results archive:', RESULTS_ZIP)
print('Checkpoint, inference code, metrics, prediction CSVs, and visual reports are included.')
if IN_COLAB and RUN_MODE != 'smoke':
    from google.colab import files
    files.download(str(RESULTS_ZIP))